## Import library

In [1]:
import os
import re
import json
import time
import random
import requests
import pandas as pd

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from pymongo import MongoClient, UpdateOne
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

## Making a MongoDB connection

In [2]:
# ============================================================
# PROJECT: News Crawling and Clustering
# STEP: 01 - News Crawling
# SOURCE: Detik.com
# ============================================================

KEYWORDS = ["jokowi", "prabowo"]

PAGE_START = 1
PAGE_END = 50

OUTPUT_DIR = "../data"
JSON_PATH = os.path.join(OUTPUT_DIR, "news_data.json")
CSV_PATH = os.path.join(OUTPUT_DIR, "news_data.csv")

MONGO_URI = os.getenv(
    "MONGO_URI",
    "mongodb://localhost:27017/"
)
DATABASE_NAME = "big_data"
COLLECTION_NAME = "news_data"

BASE_URL = "https://news.detik.com"
INDEX_URL = "https://news.detik.com/indeks"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
# HTTP session with retry
session = requests.Session()

retry_strategy = Retry(
    total=3,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)

adapter = HTTPAdapter(max_retries=retry_strategy)

session.mount("https://", adapter)
session.mount("http://", adapter)

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0.0.0 Safari/537.36"
    )
})

In [4]:
client = MongoClient(
    MONGO_URI,
    serverSelectionTimeoutMS=5000
)

# Check connection to MongoDB server
client.admin.command("ping")

db = client[DATABASE_NAME]
collection = db[COLLECTION_NAME]

# Create unique index to prevent duplicate URLs
collection.create_index("url", unique=True)

print("Koneksi MongoDB berhasil.")
print("Database:", DATABASE_NAME)
print("Collection:", COLLECTION_NAME)

Koneksi MongoDB berhasil.
Database: big_data
Collection: news_data


## Function to get a page

In [5]:
def get_soup(url):
    """
    Takes a URL and returns a BeautifulSoup object if the request is successful and the content type is HTML. Otherwise, it returns None.
    """
    try:
        response = session.get(url, timeout=20)
        response.raise_for_status()

        if "text/html" not in response.headers.get(
            "Content-Type", ""
        ):
            return None

        return BeautifulSoup(
            response.text,
            "html.parser"
        )

    except requests.RequestException as error:
        print(f"Failed to access {url}: {error}")
        return None

## Article URL validation and normalization

In [6]:
def normalize_article_url(href):
    """
    Takes a relative URL and converts it to an absolute URL, validating that it belongs to the Detik.com news domain.
    """
    if not href:
        return None

    url = urljoin(BASE_URL, href.strip())
    parsed = urlparse(url)

    allowed_hosts = {
        "news.detik.com"
    }

    if parsed.scheme not in {"http", "https"}:
        return None

    if parsed.netloc.lower() not in allowed_hosts:
        return None

    if not parsed.path or parsed.path == "/":
        return None

    # Remove fragment and query string
    clean_url = parsed._replace(
        scheme="https",
        query="",
        fragment=""
    ).geturl()

    return clean_url

## Article information extraction

In [7]:
def extract_article(url, page_number):
    """
    Extracts the title and description of an article.
    Meta description is used first, then
    article paragraphs as a fallback.
    """
    soup = get_soup(url)

    if soup is None:
        return None

    # Extracting the title
    title = None

    og_title = soup.find(
        "meta",
        property="og:title"
    )

    if og_title:
        title = og_title.get("content", "").strip()

    if not title and soup.find("h1"):
        title = soup.find("h1").get_text(
            " ",
            strip=True
        )

    # Extracting the description
    description = None

    meta_description = soup.find(
        "meta",
        attrs={"name": "description"}
    )

    if meta_description:
        description = meta_description.get(
            "content",
            ""
        ).strip()

    if not description:
        og_description = soup.find(
            "meta",
            property="og:description"
        )

        if og_description:
            description = og_description.get(
                "content",
                ""
            ).strip()

    # Fallback: extracting paragraphs from the article page
    if not description:
        paragraphs = soup.find_all("p")

        for paragraph in paragraphs:
            text = paragraph.get_text(
                " ",
                strip=True
            )

            if len(text) >= 50:
                description = text
                break

    if not title or not description:
        return None

    # Cleaning whitespace
    title = re.sub(r"\s+", " ", title).strip()
    description = re.sub(
        r"\s+",
        " ",
        description
    ).strip()

    return {
        "title": title,
        "url": url,
        "source": "detik.com",
        "description": description,
        "page_number": page_number
    }

## Crawling based on keywords

In [8]:
def scrape_news(
    keywords,
    page_start=1,
    page_end=50
):
    """
    Extracts articles from the Detik.com index page.
    Articles are saved if their title or URL contains
    any of the specified keywords.
    """
    articles_by_url = {}

    for page_number in range(
        page_start,
        page_end + 1
    ):
        page_url = f"{INDEX_URL}?page={page_number}"

        print(
            f"\nExtracting page {page_number}: "
            f"{page_url}"
        )

        soup = get_soup(page_url)

        if soup is None:
            continue

        article_urls = set()

        for link in soup.find_all("a", href=True):
            article_url = normalize_article_url(
                link["href"]
            )

            if article_url is None:
                continue

            link_text = link.get_text(
                " ",
                strip=True
            ).lower()

            # Filtering links that match the keywords
            if any(
                keyword.lower() in link_text
                for keyword in keywords
            ):
                article_urls.add(article_url)

        print(
            "Candidate links:",
            len(article_urls)
        )

        for article_url in article_urls:
            if article_url in articles_by_url:
                continue

            article = extract_article(
                article_url,
                page_number
            )

            if article is None:
                continue

            # Keywords are also checked in the title/description
            searchable_text = (
                article["title"] + " " +
                article["description"]
            ).lower()

            if any(
                keyword.lower() in searchable_text
                for keyword in keywords
            ):
                articles_by_url[article_url] = article

                print("Artikel:", article["title"])

            # Reducing the load on the server
            time.sleep(random.uniform(0.5, 1.5))

        time.sleep(random.uniform(1, 2))

    return list(articles_by_url.values())

## Running crawling

In [9]:
articles = scrape_news(
    keywords=KEYWORDS,
    page_start=PAGE_START,
    page_end=PAGE_END
)

print("\nCrawling finished.")
print("Number of unique articles:", len(articles))


Extracting page 1: https://news.detik.com/indeks?page=1
Candidate links: 1
Artikel: 2 Tahun Pemerintahan Prabowo, Hanura Ingatkan Visi Besar MBG hingga Kopdes

Extracting page 2: https://news.detik.com/indeks?page=2
Candidate links: 0

Extracting page 3: https://news.detik.com/indeks?page=3
Candidate links: 0

Extracting page 4: https://news.detik.com/indeks?page=4
Candidate links: 1
Artikel: Jokowi Sebut Polemik Ijazah Operasi Politik, PSI: Sulit untuk Tak Membenarkan

Extracting page 5: https://news.detik.com/indeks?page=5
Candidate links: 0

Extracting page 6: https://news.detik.com/indeks?page=6
Candidate links: 0

Extracting page 7: https://news.detik.com/indeks?page=7
Candidate links: 1
Artikel: Andre Rosiade: Reaktivasi Rel KA Dimulai 27 Oktober, Prabowo Komit Bangun Sumbar

Extracting page 8: https://news.detik.com/indeks?page=8
Candidate links: 0

Extracting page 9: https://news.detik.com/indeks?page=9
Candidate links: 0

Extracting page 10: https://news.detik.com/indeks?page

In [10]:
df_articles = pd.DataFrame(articles)

if not df_articles.empty:
    display(df_articles.head())
else:
    print(
        "Dataset is empty. Please check the index URL, "
        "page structure, and keywords."
    )

,title,url,source,description,page_number
0,"2 Tahun Pemerintahan Prabowo, Hanura Ingatkan ...",https://news.detik.com/berita/d-8722663/2-tahu...,detik.com,Wakil Ketua Umum Hanura Patrice Rio Capella me...,1
1,"Jokowi Sebut Polemik Ijazah Operasi Politik, P...",https://news.detik.com/berita/d-8722278/jokowi...,detik.com,PSI menanggapi pernyataan Jokowi yang menyebut...,4
2,Andre Rosiade: Reaktivasi Rel KA Dimulai 27 Ok...,https://news.detik.com/berita/d-8721755/andre-...,detik.com,Reaktivasi jalur kereta api di Sumbar dimulai ...,7
3,"Prabowo Bakal Mainkan Pemain Cadangan, PAN Taf...",https://news.detik.com/berita/d-8720450/prabow...,detik.com,PAN menilai Presiden Prabowo Subianto mengibar...,16
4,Dedi Mulyadi Buka Suara Dicari Prabowo karena ...,https://news.detik.com/berita/d-8720463/dedi-m...,detik.com,Gubernur Jabar Dedi Mulyadi menjelaskan ketida...,16


## Cleaning and exporting the dataset

In [11]:
if df_articles.empty:
    raise ValueError(
        "No articles found. "
        "Export cancelled."
    )

# Deleting empty and duplicate URLs
df_articles = df_articles.dropna(
    subset=["url", "title", "description"]
)

df_articles["url"] = df_articles["url"].astype(str)
df_articles["title"] = df_articles["title"].astype(str)
df_articles["description"] = (
    df_articles["description"].astype(str)
)

df_articles = df_articles[
    (df_articles["url"].str.strip() != "") &
    (df_articles["title"].str.strip() != "") &
    (df_articles["description"].str.strip() != "")
]

df_articles = df_articles.drop_duplicates(
    subset=["url"]
).reset_index(drop=True)

# Export JSON in the form of an array of objects
df_articles.to_json(
    JSON_PATH,
    orient="records",
    force_ascii=False,
    indent=4
)

# Ekspor CSV
df_articles.to_csv(
    CSV_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Dataset successfully exported.")
print("JSON:", JSON_PATH)
print("CSV:", CSV_PATH)
print("Number of articles:", len(df_articles))

Dataset successfully exported.
JSON: ../data\news_data.json
CSV: ../data\news_data.csv
Number of articles: 56


## Saving articles to MongoDB

In [12]:
if not df_articles.empty:
    operations = []

    for article in df_articles.to_dict(
        orient="records"
    ):
        operations.append(
            UpdateOne(
                {"url": article["url"]},
                {"$set": article},
                upsert=True
            )
        )

    if operations:
        result = collection.bulk_write(
            operations,
            ordered=False
        )

        print("MongoDB successfully updated.")
        print("New documents:", result.upserted_count)
        print("Updated documents:", result.modified_count)

print(
    "Total documents in collection:",
    collection.count_documents({})
)

MongoDB successfully updated.
New documents: 1
Updated documents: 55
Total documents in collection: 57


## Verify the crawling results

In [13]:
print("Number of rows:", len(df_articles))
print("Number of unique URLs:", df_articles["url"].nunique())
print("\nNumber of empty values:")
print(df_articles.isnull().sum())

display(
    df_articles[
        [
            "title",
            "url",
            "source",
            "description",
            "page_number"
        ]
    ].head(10)
)

Number of rows: 56
Number of unique URLs: 56

Number of empty values:
title          0
url            0
source         0
description    0
page_number    0
dtype: int64


,title,url,source,description,page_number
0,"2 Tahun Pemerintahan Prabowo, Hanura Ingatkan ...",https://news.detik.com/berita/d-8722663/2-tahu...,detik.com,Wakil Ketua Umum Hanura Patrice Rio Capella me...,1
1,"Jokowi Sebut Polemik Ijazah Operasi Politik, P...",https://news.detik.com/berita/d-8722278/jokowi...,detik.com,PSI menanggapi pernyataan Jokowi yang menyebut...,4
2,Andre Rosiade: Reaktivasi Rel KA Dimulai 27 Ok...,https://news.detik.com/berita/d-8721755/andre-...,detik.com,Reaktivasi jalur kereta api di Sumbar dimulai ...,7
3,"Prabowo Bakal Mainkan Pemain Cadangan, PAN Taf...",https://news.detik.com/berita/d-8720450/prabow...,detik.com,PAN menilai Presiden Prabowo Subianto mengibar...,16
4,Dedi Mulyadi Buka Suara Dicari Prabowo karena ...,https://news.detik.com/berita/d-8720463/dedi-m...,detik.com,Gubernur Jabar Dedi Mulyadi menjelaskan ketida...,16
5,"Bongkar Kasus 1 Ton Narkoba, Kapolri: Hadiah 2...",https://news.detik.com/berita/d-8720333/bongka...,detik.com,Kapolri Jenderal Suyudi mengungkap penyelundup...,18
6,Mekeng Nilai Kebijakan Energi Pemerintahan Pra...,https://news.detik.com/berita/d-8720330/mekeng...,detik.com,Ketua Komisi XII DPR RI Melchias Mekeng nilai ...,18
7,"Voxpol Center: 63,3% Responden Nilai Prabowo M...",https://news.detik.com/berita/d-8719656/voxpol...,detik.com,Voxpol Center Research and Consulting merilis ...,23
8,AMI Apresiasi Perhatian Prabowo pada Museum da...,https://news.detik.com/berita/d-8719680/ami-ap...,detik.com,AMI mengapresiasi perhatian Presiden Prabowo S...,23
9,Canda Prabowo ke Bos BI dan Ketua OJK: Yang At...,https://news.detik.com/berita/d-8719549/canda-...,detik.com,Presiden Prabowo Subianto mengatakan yang meng...,24
